# Joint scaling: every reservoir x every head

The full grid behind the ladders: reservoir sizes crossed with head sizes, on
NSL-KDD and K-MNIST, with the dials / RFF / raw controls and the hybrid readouts
(`qdials` = [quantum | dials], `rffdials` = [rff | dials]).

Data: `runs/joint_scaling` and `runs/joint_scaling_modes` (the first sweep,
seed 42), plus `runs/supervised_grid` once `configs/supervised_grid.yaml` has
been run.

In [ ]:
# Setup: make the repo importable from notebooks/ (or from the repo root).
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
sys.path.insert(0, str(REPO))

import pandas as pd
from elements import config
from elements.eval.results import load_results
from elements.experiments.runner import plan, run
from elements.viz import scaling
pd.set_option("display.precision", 4)
pd.set_option("display.width", 160)

In [ ]:
RUNS = [REPO / "runs" / r for r in ("joint_scaling", "supervised_grid") if (REPO / "runs" / r).exists()]
DF = load_results(*RUNS)
FIGS = REPO / "runs" / "joint_scaling" / "figs"
print(RUNS, len(DF), "rows")

## 1. Does a bigger head lift the plateau?

In [ ]:
scaling.quantum_by_head(DF, "macro_f1", out=FIGS / "quantum_by_head.png");

In [ ]:
scaling.grid_heatmap(DF, "macro_f1", out=FIGS / "quantum_grid.png");

## 2. Fair gains under the same head

In [ ]:
scaling.gain_heatmap(DF, "macro_f1", "dials", out=FIGS / "gain_vs_dials.png");
scaling.gain_heatmap(DF, "macro_f1", "rff", out=FIGS / "gain_vs_rff.png");

In [ ]:
for cfg in ("3p12m", "4p24m"):
    scaling.arms_vs_head(DF, "macro_f1", cfg, out=FIGS / f"arms_vs_head_{cfg}.png")
    scaling.arms_vs_head(DF, "macro_f1", cfg, arms=("qdials", "rffdials", "quantum", "dials"),
                         out=FIGS / f"hybrid_{cfg}.png");

## 3. More modes, few photons (K-MNIST)

In [ ]:
MODES = load_results(REPO / "runs" / "joint_scaling_modes", *RUNS)
scaling.vs_modes(MODES[MODES.dataset == "kmnist"], "macro_f1", heads=["linear", "mlp512x2"],
                 out=FIGS / "vs_modes_kmnist.png");

## 4. Best per feature view

In [ ]:
scaling.best_table(DF, "macro_f1")